In [1]:
import pandas as pd
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name='qf_21')
print(df_qf.columns)

Index(['Length of prismatic phase', 'mean AtomicRadius', 'Zr fraction',
       'Fraction of prismatic phase', 'Width of prismatic phase',
       'frac f valence electrons', 'Diameter of prismatic phase',
       'Diameter of basal phase', 'Width of basal phase',
       'Calculated Grain Boundary', 'Interant electrons',
       'Calculated Yield Strength', 'Length of Basal phase', 'Habit Plane',
       'Fraction of basal phase', 'Yang omega',
       'Distribution of precipitation', 'Grain Size', '屈服强度', '抗拉强度 （UTS）',
       '追踪编号'],
      dtype='object')


In [6]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import os
import warnings
from sklearn.exceptions import InconsistentVersionWarning

warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

# 通用函数：单变量分析并保存结果
def analyze_and_save_results(df, X_test, y_test, folder_path, weights, output_file, target_name, target_col_name, feature_ranges):
    sample_indices = range(0,X_test.shape[0],2)
    if not os.path.exists(output_file):
        df = pd.DataFrame().to_excel(output_file)
    else:
        print(f'{output_file}文件已存在')
    with pd.ExcelWriter(output_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        for idx, (feature, ranges) in enumerate(feature_ranges.items(), 1):
            print(f"Processing feature '{feature}' ({idx}/{len(feature_ranges)})...")
            results = []
            for sample_index in sample_indices:
                # if sample_index//50==0:
                #     print(f'处理sample{sample_index//50}-{sample_index//50+50}个样本')
                sample = X_test.iloc[sample_index].copy()
                track_number = df['追踪编号'].iloc[sample_index]
                max_change = 0
                max_change_value = None
                max_change_prediction = None
                previous_prediction = None
                for value in ranges:
                    modified_sample = sample.copy()
                    modified_sample[feature] = value
                    modified_sample_df = pd.DataFrame([modified_sample], columns=X_test.columns)

                    prediction_for_this_value = np.zeros(1)
                    for model_name, weight in weights.items():
                        for i in range(0, 5):
                            model_path = f'{folder_path}/{model_name}_{i}.pkl'
                            model = joblib.load(model_path)
                            prediction_for_this_value += model.predict(modified_sample_df) * weight

                    if previous_prediction is not None:
                        change = abs(prediction_for_this_value[0] - previous_prediction)
                        if change > max_change:
                            max_change = change
                            max_change_value = value
                            max_change_prediction = prediction_for_this_value[0]

                    previous_prediction = prediction_for_this_value[0]

                results.append((track_number, max_change_value, max_change_prediction))
            print(f"Processing feature '{feature}' completed.")

            results_df = pd.DataFrame(results, columns=['Track Number', f'{feature} with Max Change', f'Predicted {target_name}'])
            sheet_name = f'{feature}_max_change_{target_name}'
            results_df.to_excel(writer, sheet_name=sheet_name, index=False)

            plt.figure(figsize=(10, 6))
            plt.hist(results_df[f'{feature} with Max Change'], bins=20, edgecolor='black')
            plt.xlabel(f'{feature} with Max Change')
            plt.ylabel('Frequency')
            plt.title(f'Histogram of {feature} with Max Change ({target_name})')
            plt.grid(True)
            plt.savefig(f'{feature}_max_change_histogram_{target_name}.png')
            plt.close()

    print(f"Multi-feature single variable analysis for {target_name} completed and results saved to Excel and histograms.")

# 加载数据
df_qf = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name='qf_21')
X_test_qf = df_qf.iloc[:, :-3]
y_test_qf = df_qf['Yield_Strength']

df_kl = pd.read_excel('20240607FULL_final.xlsx', index_col=0, sheet_name='kl_21')
X_test_kl = df_kl.iloc[:, :-3]
y_test_kl = df_kl['Tensile_Strength （UTS）']

# 定义模型路径和权重
folder_path_qf = "small_model/final_model/qf/rf"
weights_qf = {'rf': 0.2}
folder_path_kl = "small_model/final_model/kl/rf"
weights_kl = {'rf': 0.2}

# 小模型特征及其变化范围
small_model_feature_ranges = {
    'Diameter of basal phase': np.arange(1, 100, 1),
    'Width of basal phase': np.arange(0.5, 20, 0.2),
    'Diameter of prismatic phase': np.arange(0, 31, 0.5),
    'Fraction of basal phase':np.arange(0,20,0.5),
    'Fraction of prismatic phase':np.arange(0,60,1),
    'Width of prismatic phase':np.arange(0,200,2)
}

# 分析并保存屈服强度和抗拉强度的结果（小模型）
# analyze_and_save_results(df_qf, X_test_qf, y_test_qf, folder_path_qf, weights_qf, 'small_qf_multiple_single_variable_analysis.xlsx', 'Yield Strength', 'Yield_Strength', small_model_feature_ranges)
# analyze_and_save_results(df_kl, X_test_kl, y_test_kl, folder_path_kl, weights_kl, 'small_kl_multiple_single_variable_analysis.xlsx', 'Tensile Strength', 'Tensile_Strength （UTS）', small_model_feature_ranges)

# 通用模型数据加载
df_qf_generic = pd.read_excel('general_model/qf_models/train_set_new.xlsx', index_col=0)
df_kl_generic = pd.read_excel('general_model/kl_models/train_set_new.xlsx', index_col=0)


track_numbers_qf_generic = df_qf_generic['追踪编号']
X_test_qf_generic = df_qf_generic.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_qf_generic = df_qf_generic['Yield_Strength']

track_numbers_kl_generic = df_kl_generic['追踪编号']
X_test_kl_generic = df_kl_generic.drop(columns=['Precipitate Distribution', 'Yield_Strength', 'Tensile_Strength (UTS)', '追踪编号'])
y_test_kl_generic = df_kl_generic['Tensile_Strength (UTS)']

# 定义模型路径和权重（通用模型）
folder_path_qf_generic = "general_model/qf_models"
folder_path_kl_generic = "general_model/kl_models"
weights_qf_generic = {'xgboost': 0.7 / 5, 'rf': 0.3 / 5}
weights_kl_generic = {'xgboost': 0.5 / 5, 'rf': 0.5 / 5}


generic_model_feature_ranges1 = {
    'Interant electrons':np.arange(1, 19, 1),
    'Interant d electrons':np.arange(1, 9, 1)
}
# 通用模型特征及其变化范围
generic_model_feature_ranges = {
    'Calculated Yield Strength': np.arange(70, 250, 5),
    'Grain Size': np.arange(5, 31, 1),
    'Y fraction': np.arange(0.005, 0.04, 0.002),
    'Gd fraction': np.arange(0.005, 0.04, 0.002),
    'Interant electrons':np.arange(1, 19, 1),
    'Interant d electrons':np.arange(1, 9, 1)
}

# # 分析并保存屈服强度和抗拉强度的结果（通用模型）
# analyze_and_save_results(df_qf_generic, X_test_qf_generic, y_test_qf_generic, folder_path_qf_generic, weights_qf_generic, 'generic_qf_multiple_single_variable_analysis_electrons.xlsx', 'Yield Strength', 'Yield_Strength', generic_model_feature_ranges1)
analyze_and_save_results(df_kl_generic, X_test_kl_generic, y_test_kl_generic, folder_path_kl_generic, weights_kl_generic, 'generic_kl_multiple_single_variable_analysis.xlsx', 'Tensile Strength', 'Tensile_Strength (UTS)', generic_model_feature_ranges)

print("Multi-feature single variable analysis for both models completed and results saved to Excel and histograms.")


generic_kl_multiple_single_variable_analysis.xlsx文件已存在
Processing feature 'Calculated Yield Strength' (1/6)...
Processing feature 'Calculated Yield Strength' completed.
Processing feature 'Grain Size' (2/6)...


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Processing feature 'Grain Size' completed.
Processing feature 'Y fraction' (3/6)...


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Processing feature 'Y fraction' completed.
Processing feature 'Gd fraction' (4/6)...


...

Processing feature 'Interant electrons' completed.
Processing feature 'Interant d electrons' (6/6)...


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Processing feature 'Interant d electrons' completed.


F:\Anaconda\envs\new_env\lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Multi-feature single variable analysis for Tensile Strength completed and results saved to Excel and histograms.
Multi-feature single variable analysis for both models completed and results saved to Excel and histograms.


In [6]:
# 处理方式一、计算阈值出现的频次和对应的变化
import pandas as pd

file_path = 'generic_qf_multiple_single_variable_analysis.xlsx'
excel_data = pd.ExcelFile(file_path)
sheet_names = excel_data.sheet_names
print(sheet_names)
def process_sheet(sheet_name,target_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    sensitive_points = data[f'{sheet_name} with Max Change']
    predicted_strengths = data[target_name]

    avg_predicted_strengths = predicted_strengths.groupby(sensitive_points).mean()
    proportion_sensitive_points = sensitive_points.value_counts(normalize=True)

    result_df = pd.DataFrame({
        f'Average {target_name}': avg_predicted_strengths,
        'Proportion': proportion_sensitive_points
    }).reset_index()
    result_df.rename(columns={'index': 'Sensitive Point'}, inplace=True)
    return result_df

processed_data = {sheet_name: process_sheet(sheet_name,'Predicted Yield Strength') for sheet_name in sheet_names if sheet_name!='Sheet1'}

output_file_path = 'processed_yield_strength_analysis.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    for sheet_name, result_df in processed_data.items():
        result_df.to_excel(writer, sheet_name=sheet_name, index=False)

# 处理方式二、筛选出现出量较多的特征，分组绘图分析

['Sheet1', 'Calculated Yield Strength', 'Grain Size', 'Y fraction', 'Gd fraction']


In [20]:
import pandas as pd

# Load the Excel file
file_path = 'generic_qf_multiple_single_variable_analysis.xlsx'
excel_data = pd.ExcelFile(file_path)

# Function to process each sheet for the second requirement
def process_sheet_second(sheet_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    sensitive_points_col = f'{sheet_name} with Max Change'
    sensitive_points = data[sensitive_points_col]
    
    # Calculate the frequency of each sensitive point
    frequency_sensitive_points = sensitive_points.value_counts()
    
    # Filter out points with frequency less than 5%
    total_points = len(sensitive_points)
    filtered_points = frequency_sensitive_points[frequency_sensitive_points / total_points >= 0.05]
    
    # Create a new DataFrame for the filtered results
    filtered_df = data[data[sensitive_points_col].isin(filtered_points.index)]
    
    # Sort the filtered points to handle them in order
    sorted_points = filtered_points.index.sort_values()
    
    # Initialize a list to store renamed points and their boundaries
    renamed_intervals = []

    # Function to find an appropriate interval name
    def find_interval(point, renamed_intervals):
        for start, end in renamed_intervals:
            if start <= point <= end:
                return f"{start:.2f} to {end:.2f}"
        return None
    
    # Rename the sensitive points based on the relative condition (within 10% difference)
    for point in sorted_points:
        interval_name = find_interval(point, renamed_intervals)
        if not interval_name:
            start = point
            end = point * 1.10
            renamed_intervals.append((start, end))
            interval_name = f"{start:.2f} to {end:.2f}" if start != end else f"{start:.2f}"
        
        # Update the sensitive points with the new interval name
        filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
    
    # Calculate the proportion of each renamed sensitive point in the original dataset
    renamed_point_proportions = filtered_df['Renamed Sensitive Point'].value_counts() / total_points
    
    # Map the proportions back to the filtered DataFrame
    filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
    
    return filtered_df[['Track Number', 'Renamed Sensitive Point', 'Predicted Yield Strength', 'Proportion in Original Data']]

# Process each sheet
processed_data_second = {sheet_name: process_sheet_second(sheet_name) for sheet_name in excel_data.sheet_names if sheet_name != 'Sheet1'}

# Save the results to a new Excel file
output_file_path_second = 'processed_general_model_yield_strength_analysis_second.xlsx'
with pd.ExcelWriter(output_file_path_second) as writer:
    for sheet_name, result_df in processed_data_second.items():
        result_df.to_excel(writer, sheet_name=sheet_name, index=False)


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2516\1497483082.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2516\1497483082.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_2516\1497483082.py

In [3]:
import pandas as pd

# Load the Excel file
file_path = 'generic_kl_multiple_single_variable_analysis.xlsx'
excel_data = pd.ExcelFile(file_path)

# Function to process each sheet for the second requirement
def process_sheet_second(sheet_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    sensitive_points_col = f'{sheet_name} with Max Change'
    sensitive_points = data[sensitive_points_col]
    
    # Calculate the frequency of each sensitive point
    frequency_sensitive_points = sensitive_points.value_counts()
    
    # Filter out points with frequency less than 5%
    total_points = len(sensitive_points)
    filtered_points = frequency_sensitive_points[frequency_sensitive_points / total_points >= 0.05]
    
    # Create a new DataFrame for the filtered results
    filtered_df = data[data[sensitive_points_col].isin(filtered_points.index)]
    
    # Sort the filtered points to handle them in order
    sorted_points = filtered_points.index.sort_values()
    
    # Initialize a list to store renamed points and their boundaries
    renamed_intervals = []

    # Function to find an appropriate interval name
    def find_interval(point, renamed_intervals):
        for start, end in renamed_intervals:
            if start <= point <= end:
                return f"{start:.2f} to {end:.2f}"
        return None
    
    # Rename the sensitive points based on the relative condition (within 10% difference)
    for point in sorted_points:
        interval_name = find_interval(point, renamed_intervals)
        if not interval_name:
            start = point
            end = point * 1.10
            renamed_intervals.append((start, end))
            interval_name = f"{start:.2f} to {end:.2f}" if start != end else f"{start:.2f}"
        
        # Update the sensitive points with the new interval name
        filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
    
    # Calculate the proportion of each renamed sensitive point in the original dataset
    renamed_point_proportions = filtered_df['Renamed Sensitive Point'].value_counts() / total_points
    
    # Map the proportions back to the filtered DataFrame
    filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
    
    return filtered_df[['Track Number', 'Renamed Sensitive Point', 'Predicted Tensile Strength', 'Proportion in Original Data']]

# Process each sheet
processed_data_second = {sheet_name: process_sheet_second(sheet_name) for sheet_name in excel_data.sheet_names if sheet_name != 'Sheet1'}

# Save the results to a new Excel file
output_file_path_second = 'processed_general_model_tensile_strength_analysis_second.xlsx'
with pd.ExcelWriter(output_file_path_second) as writer:
    for sheet_name, result_df in processed_data_second.items():
        result_df.to_excel(writer, sheet_name=sheet_name, index=False)


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_23120\773271592.py:46: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_23120\773271592.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_23120\773271592.py

In [4]:
import pandas as pd

# Load the Excel file
file_path = 'small_qf_multiple_single_variable_analysis.xlsx'
excel_data = pd.ExcelFile(file_path)

# Function to process each sheet for the second requirement
def process_sheet_second(sheet_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    sensitive_points_col = f'{sheet_name} with Max Change'
    sensitive_points = data[sensitive_points_col]
    
    # Calculate the frequency of each sensitive point
    frequency_sensitive_points = sensitive_points.value_counts()
    
    # Filter out points with frequency less than 5%
    total_points = len(sensitive_points)
    filtered_points = frequency_sensitive_points[frequency_sensitive_points / total_points >= 0.02]
    
    # Create a new DataFrame for the filtered results
    filtered_df = data[data[sensitive_points_col].isin(filtered_points.index)]
    
    # Sort the filtered points to handle them in order
    sorted_points = filtered_points.index.sort_values()
    
    # Initialize a list to store renamed points and their boundaries
    renamed_intervals = []

    # Function to find an appropriate interval name
    def find_interval(point, renamed_intervals):
        for start, end in renamed_intervals:
            if start <= point <= end:
                return f"{start:.2f} to {end:.2f}"
        return None
    
    # Rename the sensitive points based on the relative condition (within 10% difference)
    for point in sorted_points:
        interval_name = find_interval(point, renamed_intervals)
        if not interval_name:
            start = point
            end = point * 1.10
            renamed_intervals.append((start, end))
            interval_name = f"{start:.2f} to {end:.2f}" if start != end else f"{start:.2f}"
        
        # Update the sensitive points with the new interval name
        filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
    
    # Calculate the proportion of each renamed sensitive point in the original dataset
    renamed_point_proportions = filtered_df['Renamed Sensitive Point'].value_counts() / total_points
    
    # Map the proportions back to the filtered DataFrame
    filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
    
    return filtered_df[['Track Number', 'Renamed Sensitive Point', 'Predicted Yield Strength', 'Proportion in Original Data']]

# Process each sheet
processed_data_second = {sheet_name: process_sheet_second(sheet_name) for sheet_name in excel_data.sheet_names if sheet_name != 'Sheet1'}

# Save the results to a new Excel file
output_file_path_second = 'processed_small_model_yield_strength_analysis_second.xlsx'
with pd.ExcelWriter(output_file_path_second) as writer:
    for sheet_name, result_df in processed_data_second.items():
        result_df.to_excel(writer, sheet_name=sheet_name, index=False)


In [ ]:
import pandas as pd

# Load the Excel file
file_path = 'small_kl_multiple_single_variable_analysis.xlsx'
excel_data = pd.ExcelFile(file_path)

# Function to process each sheet for the second requirement
def process_sheet_second(sheet_name):
    data = pd.read_excel(file_path, sheet_name=sheet_name)
    sensitive_points_col = f'{sheet_name} with Max Change'
    sensitive_points = data[sensitive_points_col]
    
    # Calculate the frequency of each sensitive point
    frequency_sensitive_points = sensitive_points.value_counts()
    
    # Filter out points with frequency less than 5%
    total_points = len(sensitive_points)
    filtered_points = frequency_sensitive_points[frequency_sensitive_points / total_points >= 0.02]
    
    # Create a new DataFrame for the filtered results
    filtered_df = data[data[sensitive_points_col].isin(filtered_points.index)]
    
    # Sort the filtered points to handle them in order
    sorted_points = filtered_points.index.sort_values()
    
    # Initialize a list to store renamed points and their boundaries
    renamed_intervals = []

    # Function to find an appropriate interval name
    def find_interval(point, renamed_intervals):
        for start, end in renamed_intervals:
            if start <= point <= end:
                return f"{start:.2f} to {end:.2f}"
        return None
    
    # Rename the sensitive points based on the relative condition (within 10% difference)
    for point in sorted_points:
        interval_name = find_interval(point, renamed_intervals)
        if not interval_name:
            start = point
            end = point * 1.10
            renamed_intervals.append((start, end))
            interval_name = f"{start:.2f} to {end:.2f}" if start != end else f"{start:.2f}"
        
        # Update the sensitive points with the new interval name
        filtered_df.loc[filtered_df[sensitive_points_col] == point, 'Renamed Sensitive Point'] = interval_name
    
    # Calculate the proportion of each renamed sensitive point in the original dataset
    renamed_point_proportions = filtered_df['Renamed Sensitive Point'].value_counts() / total_points
    
    # Map the proportions back to the filtered DataFrame
    filtered_df['Proportion in Original Data'] = filtered_df['Renamed Sensitive Point'].map(renamed_point_proportions)
    
    return filtered_df[['Track Number', 'Renamed Sensitive Point', 'Predicted Tensile Strength', 'Proportion in Original Data']]

# Process each sheet
processed_data_second = {sheet_name: process_sheet_second(sheet_name) for sheet_name in excel_data.sheet_names if sheet_name != 'Sheet1'}

# Save the results to a new Excel file
output_file_path_second = 'processed_small_model_tensile_strength_analysis_second.xlsx'
with pd.ExcelWriter(output_file_path_second) as writer:
    for sheet_name, result_df in processed_data_second.items():
        result_df.to_excel(writer, sheet_name=sheet_name, index=False)
